# Titanic competition with TensorFlow Decision Forests

This notebook will take you through the steps needed to train a baseline Gradient Boosted Trees Model using TensorFlow Decision Forests and creating a submission on the Titanic competition. 

This notebook shows:

1. How to do some basic pre-processing. For example, the passenger names will be tokenized, and ticket names will be splitted in parts.
1. How to train a Gradient Boosted Trees (GBT) with default parameters
1. How to train a GBT with improved default parameters
1. How to tune the parameters of a GBTs
1. How to train and ensemble many GBTs

# Imports dependencies

# 

In [1]:
import numpy as np
import pandas as pd
import os

import tensorflow as tf
import tensorflow_decision_forests as tfdf

print(f"Found TF-DF {tfdf.__version__}")

Found TF-DF 1.2.0


# Load dataset

In [2]:
train_df = pd.read_csv("/kaggle/input/titanic/train.csv")
serving_df = pd.read_csv("/kaggle/input/titanic/test.csv")

train_df.head(10)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
5,6,0,3,"Moran, Mr. James",male,NaN,0,0,330877,8.4583,NaN,Q
6,7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463,51.8625,E46,S
7,8,0,3,"Palsson, Master. Gosta Leonard",male,2.0,3,1,349909,21.0750,NaN,S
8,9,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,27.0,0,2,347742,11.1333,NaN,S
9,10,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",female,14.0,1,0,237736,30.0708,NaN,C


# Prepare dataset

We will apply the following transformations on the dataset.

1. Tokenize the names. For example, "Braund, Mr. Owen Harris" will become ["Braund", "Mr.", "Owen", "Harris"].
2. Extract any prefix in the ticket. For example ticket "STON/O2. 3101282" will become "STON/O2." and 3101282.

In [3]:
def preprocess(df):
    df = df.copy()
    
    def normalize_name(x):
        return " ".join([v.strip(",()[].\"'") for v in x.split(" ")])
    
    def ticket_number(x):
        return x.split(" ")[-1]
        
    def ticket_item(x):
        items = x.split(" ")
        if len(items) == 1:
            return "NONE"
        return "_".join(items[0:-1])
    
    df["Name"] = df["Name"].apply(normalize_name)
    df["Ticket_number"] = df["Ticket"].apply(ticket_number)
    df["Ticket_item"] = df["Ticket"].apply(ticket_item)                     
    return df
    
preprocessed_train_df = preprocess(train_df)
preprocessed_serving_df = preprocess(serving_df)

preprocessed_train_df.head(5)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Ticket_number,Ticket_item
0,1,0,3,Braund Mr Owen Harris,male,22.0,1,0,A/5 21171,7.2500,NaN,S,21171,A/5
1,2,1,1,Cumings Mrs John Bradley Florence Briggs Thayer,female,38.0,1,0,PC 17599,71.2833,C85,C,17599,PC
2,3,1,3,Heikkinen Miss Laina,female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,3101282,STON/O2.
3,4,1,1,Futrelle Mrs Jacques Heath Lily May Peel,female,35.0,1,0,113803,53.1000,C123,S,113803,NONE
4,5,0,3,Allen Mr William Henry,male,35.0,0,0,373450,8.0500,NaN,S,373450,NONE


Let's keep the list of the input features of the model. Notably, we don't want to train our model on the "PassengerId" and "Ticket" features.

In [4]:
input_features = list(preprocessed_train_df.columns)
input_features.remove("Ticket")
input_features.remove("PassengerId")
input_features.remove("Survived")
#input_features.remove("Ticket_number")

print(f"Input features: {input_features}")

Input features: ['Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Cabin', 'Embarked', 'Ticket_number', 'Ticket_item']


# Convert Pandas dataset to TensorFlow Dataset

In [5]:
def tokenize_names(features, labels=None):
    """Divite the names into tokens. TF-DF can consume text tokens natively."""
    features["Name"] =  tf.strings.split(features["Name"])
    return features, labels

train_ds = tfdf.keras.pd_dataframe_to_tf_dataset(preprocessed_train_df,label="Survived").map(tokenize_names)
serving_ds = tfdf.keras.pd_dataframe_to_tf_dataset(preprocessed_serving_df).map(tokenize_names)

# Train model with default parameters

### Train model

First, we are training a GradientBoostedTreesModel model with the default parameters.

In [6]:
model = tfdf.keras.GradientBoostedTreesModel(
    verbose=0, # Very few logs
    features=[tfdf.keras.FeatureUsage(name=n) for n in input_features],
    exclude_non_specified_features=True, # Only use the features in "features"
    random_seed=1234,
)
model.fit(train_ds)

self_evaluation = model.make_inspector().evaluation()
print(f"Accuracy: {self_evaluation.accuracy} Loss:{self_evaluation.loss}")

[INFO 2026-03-26T10:07:47.714907738+00:00 kernel.cc:1214] Loading model from path /tmp/tmp47ujyigj/model/ with prefix c76cb9942efe4520
[INFO 2026-03-26T10:07:47.722888558+00:00 abstract_model.cc:1311] Engine "GradientBoostedTreesQuickScorerExtended" built
[INFO 2026-03-26T10:07:47.72296135+00:00 kernel.cc:1046] Use fast generic engine


Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: could not get source code
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert
Accuracy: 0.8260869383811951 Loss:0.8608942627906799


# Train model with improved default parameters

Now you'll use some specific parameters when creating the GBT model

In [7]:
model = tfdf.keras.GradientBoostedTreesModel(
    verbose=0, # Very few logs
    features=[tfdf.keras.FeatureUsage(name=n) for n in input_features],
    exclude_non_specified_features=True, # Only use the features in "features"
    
    #num_trees=2000,
    
    # Only for GBT.
    # A bit slower, but great to understand the model.
    # compute_permutation_variable_importance=True,
    
    # Change the default hyper-parameters
    # hyperparameter_template="benchmark_rank1@v1",
    
    #num_trees=1000,
    #tuner=tuner
    
    min_examples=1,
    categorical_algorithm="RANDOM",
    #max_depth=4,
    shrinkage=0.05,
    #num_candidate_attributes_ratio=0.2,
    split_axis="SPARSE_OBLIQUE",
    sparse_oblique_normalization="MIN_MAX",
    sparse_oblique_num_projections_exponent=2.0,
    num_trees=2000,
    #validation_ratio=0.0,
    random_seed=1234,
    
)
model.fit(train_ds)

self_evaluation = model.make_inspector().evaluation()
print(f"Accuracy: {self_evaluation.accuracy} Loss:{self_evaluation.loss}")

[INFO 2026-03-26T10:07:50.500074418+00:00 kernel.cc:1214] Loading model from path /tmp/tmpqvdtnb46/model/ with prefix 565595b4125e491a
[INFO 2026-03-26T10:07:50.508519993+00:00 decision_forest.cc:661] Model loaded with 33 root(s), 1823 node(s), and 10 input feature(s).
[INFO 2026-03-26T10:07:50.508580201+00:00 kernel.cc:1046] Use fast generic engine


Accuracy: 0.760869562625885 Loss:1.0154211521148682


Let's look at the model and you can also notice the information about variable importance that the model figured out

In [8]:
model.summary()

Model: "gradient_boosted_trees_model_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
Total params: 1
Trainable params: 0
Non-trainable params: 1
_________________________________________________________________
Type: "GRADIENT_BOOSTED_TREES"
Task: CLASSIFICATION
Label: "__LABEL"

Input Features (11):
	Age
	Cabin
	Embarked
	Fare
	Name
	Parch
	Pclass
	Sex
	SibSp
	Ticket_item
	Ticket_number

No weights

Variable Importance: INV_MEAN_MIN_DEPTH:
    1.           "Sex"  0.576632 ################
    2.           "Age"  0.364297 #######
    3.          "Fare"  0.278839 ####
    4.          "Name"  0.208548 #
    5. "Ticket_number"  0.180792 
    6.        "Pclass"  0.176962 
    7.         "Parch"  0.176659 
    8.   "Ticket_item"  0.175540 
    9.      "Embarked"  0.172339 
   10.         "SibSp"  0.170442 

Variable Importance: NUM_AS_ROOT:
    1.  "Sex" 28.000000 ################
    2. "Name"  5.000000 

# Make predictions

In [9]:
def prediction_to_kaggle_format(model, threshold=0.5):
    proba_survive = model.predict(serving_ds, verbose=0)[:,0]
    return pd.DataFrame({
        "PassengerId": serving_df["PassengerId"],
        "Survived": (proba_survive >= threshold).astype(int)
    })

def make_submission(kaggle_predictions):
    path="/kaggle/working/submission.csv"
    kaggle_predictions.to_csv(path, index=False)
    print(f"Submission exported to {path}")
    
kaggle_predictions = prediction_to_kaggle_format(model)
make_submission(kaggle_predictions)
!head /kaggle/working/submission.csv

Submission exported to /kaggle/working/submission.csv
PassengerId,Survived
892,0
893,0
894,0
895,0
896,0
897,0
898,0
899,0
900,1


# Training a model with hyperparameter tunning

Hyper-parameter tuning is enabled by specifying the tuner constructor argument of the model. The tuner object contains all the configuration of the tuner (search space, optimizer, trial and objective).


In [10]:
tuner = tfdf.tuner.RandomSearch(num_trials=1000)
tuner.choice("min_examples", [2, 5, 7, 10])
tuner.choice("categorical_algorithm", ["CART", "RANDOM"])

local_search_space = tuner.choice("growing_strategy", ["LOCAL"])
local_search_space.choice("max_depth", [3, 4, 5, 6, 8])

global_search_space = tuner.choice("growing_strategy", ["BEST_FIRST_GLOBAL"], merge=True)
global_search_space.choice("max_num_nodes", [16, 32, 64, 128, 256])

#tuner.choice("use_hessian_gain", [True, False])
tuner.choice("shrinkage", [0.02, 0.05, 0.10, 0.15])
tuner.choice("num_candidate_attributes_ratio", [0.2, 0.5, 0.9, 1.0])


tuner.choice("split_axis", ["AXIS_ALIGNED"])
oblique_space = tuner.choice("split_axis", ["SPARSE_OBLIQUE"], merge=True)
oblique_space.choice("sparse_oblique_normalization",
                     ["NONE", "STANDARD_DEVIATION", "MIN_MAX"])
oblique_space.choice("sparse_oblique_weights", ["BINARY", "CONTINUOUS"])
oblique_space.choice("sparse_oblique_num_projections_exponent", [1.0, 1.5])

# Tune the model. Notice the `tuner=tuner`.
tuned_model = tfdf.keras.GradientBoostedTreesModel(tuner=tuner)
tuned_model.fit(train_ds, verbose=0)

tuned_self_evaluation = tuned_model.make_inspector().evaluation()
print(f"Accuracy: {tuned_self_evaluation.accuracy} Loss:{tuned_self_evaluation.loss}")

Use /tmp/tmpr2nb5n2e as temporary training directory


[INFO 2026-03-26T10:09:33.458323691+00:00 kernel.cc:1214] Loading model from path /tmp/tmpr2nb5n2e/model/ with prefix 23dd3a60a91b4c79
[INFO 2026-03-26T10:09:33.470433165+00:00 decision_forest.cc:661] Model loaded with 19 root(s), 589 node(s), and 12 input feature(s).
[INFO 2026-03-26T10:09:33.470467702+00:00 abstract_model.cc:1311] Engine "GradientBoostedTreesGeneric" built
[INFO 2026-03-26T10:09:33.470490594+00:00 kernel.cc:1046] Use fast generic engine


Accuracy: 0.9178082346916199 Loss:0.6503586769104004


In the last line in the cell above, you can see the accuracy is higher than previously with default parameters and parameters set by hand.

This is the main idea behing hyperparameter tuning.

For more information you can follow this tutorial: [Automated hyper-parameter tuning](https://www.tensorflow.org/decision_forests/tutorials/automatic_tuning_colab)

# Making an ensemble

Here you'll create 100 models with different seeds and combine their results

This approach removes a little bit the random aspects related to creating ML models

In the GBT creation is used the `honest` parameter. It will use different training examples to infer the structure and the leaf values. This regularization technique trades examples for bias estimates.

In [11]:
predictions = None
num_predictions = 0

for i in range(100):
    print(f"i:{i}")
    # Possible models: GradientBoostedTreesModel or RandomForestModel
    model = tfdf.keras.GradientBoostedTreesModel(
        verbose=0, # Very few logs
        features=[tfdf.keras.FeatureUsage(name=n) for n in input_features],
        exclude_non_specified_features=True, # Only use the features in "features"

        #min_examples=1,
        #categorical_algorithm="RANDOM",
        ##max_depth=4,
        #shrinkage=0.05,
        ##num_candidate_attributes_ratio=0.2,
        #split_axis="SPARSE_OBLIQUE",
        #sparse_oblique_normalization="MIN_MAX",
        #sparse_oblique_num_projections_exponent=2.0,
        #num_trees=2000,
        ##validation_ratio=0.0,
        random_seed=i,
        honest=True,
    )
    model.fit(train_ds)
    
    sub_predictions = model.predict(serving_ds, verbose=0)[:,0]
    if predictions is None:
        predictions = sub_predictions
    else:
        predictions += sub_predictions
    num_predictions += 1

predictions/=num_predictions

kaggle_predictions = pd.DataFrame({
        "PassengerId": serving_df["PassengerId"],
        "Survived": (predictions >= 0.5).astype(int)
    })

make_submission(kaggle_predictions)

i:0


[INFO 2026-03-26T10:09:34.262374018+00:00 kernel.cc:1214] Loading model from path /tmp/tmpfqofb9oq/model/ with prefix 909407c4e6254f13
[INFO 2026-03-26T10:09:34.266527188+00:00 kernel.cc:1046] Use fast generic engine


i:1


[INFO 2026-03-26T10:09:35.377763082+00:00 kernel.cc:1214] Loading model from path /tmp/tmp694i6qqt/model/ with prefix 0fb49533d28545bc
[INFO 2026-03-26T10:09:35.396561158+00:00 kernel.cc:1046] Use fast generic engine


i:2


...

i:98


[INFO 2026-03-26T10:11:11.733175523+00:00 kernel.cc:1214] Loading model from path /tmp/tmp5s7tgq7w/model/ with prefix 02ad1e46cbcd4edc
[INFO 2026-03-26T10:11:11.740473927+00:00 kernel.cc:1046] Use fast generic engine


i:99


[INFO 2026-03-26T10:11:12.82371936+00:00 kernel.cc:1214] Loading model from path /tmp/tmpmufmu7bz/model/ with prefix 34b4f6d78e3a49d9
[INFO 2026-03-26T10:11:12.837238286+00:00 kernel.cc:1046] Use fast generic engine


Submission exported to /kaggle/working/submission.csv


In [12]:
# ================================
# SIMPLE LOGISTIC REGRESSION
# ================================
import numpy as np
import pandas as pd

# Load data
train_df = pd.read_csv("/kaggle/input/titanic/train.csv")
test_df = pd.read_csv("/kaggle/input/titanic/train.csv")

# Preprocessing
def preprocess(df):
    df = df.copy()
    df["Age"].fillna(df["Age"].median(), inplace=True)
    df["Fare"].fillna(df["Fare"].median(), inplace=True)
    df["Embarked"].fillna("S", inplace=True)

    df["Sex"] = df["Sex"].map({"male":0, "female":1})
    df = pd.get_dummies(df, columns=["Embarked"], drop_first=True)

    return df

train_df = preprocess(train_df)
test_df = preprocess(test_df)

features = ["Pclass","Sex","Age","SibSp","Parch","Fare","Embarked_Q","Embarked_S"]

X = train_df[features].values
y = train_df["Survived"].values.reshape(-1,1)

X_test = test_df[features].values

# Sigmoid
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

# Initialize
w = np.zeros((X.shape[1],1))
b = 0

# Training (manual gradient descent)
lr = 0.01
epochs = 1000

for i in range(epochs):
    z = X @ w + b
    y_hat = sigmoid(z)

    dz = y_hat - y
    dw = (1/len(X)) * (X.T @ dz)
    db = np.mean(dz)

    w -= lr * dw
    b -= lr * db

# Prediction
preds = sigmoid(X_test @ w + b)
preds = (preds >= 0.5).astype(int)

submission = pd.DataFrame({
    "PassengerId": test_df["PassengerId"],
    "Survived": preds.flatten()
})

submission.to_csv("/kaggle/working/submission_simple.csv", index=False)

In [13]:
# ================================
# JAX OPTIMIZED VERSION
# ================================
import jax
import jax.numpy as jnp
import pandas as pd

# Load + preprocess (same as before)
train_df = pd.read_csv("/kaggle/input/titanic/train.csv")
test_df = pd.read_csv("/kaggle/input/titanic/test.csv")

def preprocess(df):
    df = df.copy()
    df["Age"].fillna(df["Age"].median(), inplace=True)
    df["Fare"].fillna(df["Fare"].median(), inplace=True)
    df["Embarked"].fillna("S", inplace=True)

    df["Sex"] = df["Sex"].map({"male":0, "female":1})
    df = pd.get_dummies(df, columns=["Embarked"], drop_first=True)

    return df

train_df = preprocess(train_df)
test_df = preprocess(test_df)

features = ["Pclass","Sex","Age","SibSp","Parch","Fare","Embarked_Q","Embarked_S"]

X = jnp.array(train_df[features].values)
y = jnp.array(train_df["Survived"].values)

X_test = jnp.array(test_df[features].values)

# Model
def sigmoid(z):
    return 1 / (1 + jnp.exp(-z))

def predict(params, X):
    w, b = params
    return sigmoid(jnp.dot(X, w) + b)

# Loss
def loss_fn(params, X, y):
    preds = predict(params, X)
    return -jnp.mean(y * jnp.log(preds + 1e-8) + (1-y)*jnp.log(1-preds + 1e-8))

# Gradient
grad_fn = jax.grad(loss_fn)

# JIT compiled step
@jax.jit
def update(params, X, y, lr):
    grads = grad_fn(params, X, y)
    return (params[0] - lr * grads[0],
            params[1] - lr * grads[1])

# Initialize
params = (jnp.zeros(X.shape[1]), 0.0)

# Training
lr = 0.01
for i in range(1000):
    params = update(params, X, y, lr)

# Predictions (vmap for batch)
preds = predict(params, X_test)
preds = (preds >= 0.5).astype(int)

submission = pd.DataFrame({
    "PassengerId": test_df["PassengerId"],
    "Survived": np.array(preds)
})

submission.to_csv("/kaggle/working/submission_jax.csv", index=False)

In [14]:
# ============================================
# TITANIC: NORMAL vs JAX (jit + grad)
# ============================================

import numpy as np
import pandas as pd
import time

# JAX
import jax
import jax.numpy as jnp

# ================================
# LOAD DATA
# ================================
train_df = pd.read_csv("/kaggle/input/titanic/train.csv")
test_df = pd.read_csv("/kaggle/input/titanic/test.csv")

# ================================
# PREPROCESSING
# ================================
def preprocess(df):
    df = df.copy()

    df["Age"].fillna(df["Age"].median(), inplace=True)
    df["Fare"].fillna(df["Fare"].median(), inplace=True)
    df["Embarked"].fillna("S", inplace=True)

    df["Sex"] = df["Sex"].map({"male":0, "female":1})

    df = pd.get_dummies(df, columns=["Embarked"], drop_first=True)

    return df

train_df = preprocess(train_df)
test_df = preprocess(test_df)

features = ["Pclass","Sex","Age","SibSp","Parch","Fare","Embarked_Q","Embarked_S"]

X = train_df[features].values
y = train_df["Survived"].values.reshape(-1,1)

X_test = test_df[features].values

# ============================================
# 1. NORMAL LOGISTIC REGRESSION
# ============================================

def sigmoid_np(z):
    return 1 / (1 + np.exp(-z))

w = np.zeros((X.shape[1],1))
b = 0

lr = 0.01
epochs = 1000

start = time.time()

for i in range(epochs):
    z = X @ w + b
    y_hat = sigmoid_np(z)

    dz = y_hat - y
    dw = (1/len(X)) * (X.T @ dz)
    db = np.mean(dz)

    w -= lr * dw
    b -= lr * db

end = time.time()

normal_time = end - start

# Accuracy (train)
preds_train = (sigmoid_np(X @ w + b) >= 0.5).astype(int)
normal_acc = np.mean(preds_train == y)

# Test prediction
preds_test = (sigmoid_np(X_test @ w + b) >= 0.5).astype(int)

submission_normal = pd.DataFrame({
    "PassengerId": test_df["PassengerId"],
    "Survived": preds_test.flatten()
})

submission_normal.to_csv("/kaggle/working/submission_normal.csv", index=False)

# ============================================
# 2. JAX VERSION (jit + grad)
# ============================================

X_j = jnp.array(X)
y_j = jnp.array(y.flatten())
X_test_j = jnp.array(X_test)

def sigmoid_j(z):
    return 1 / (1 + jnp.exp(-z))

def predict(params, X):
    w, b = params
    return sigmoid_j(jnp.dot(X, w) + b)

def loss_fn(params, X, y):
    preds = predict(params, X)
    return -jnp.mean(y * jnp.log(preds + 1e-8) + (1-y)*jnp.log(1-preds + 1e-8))

grad_fn = jax.grad(loss_fn)

@jax.jit
def update(params, X, y, lr):
    grads = grad_fn(params, X, y)
    return (params[0] - lr * grads[0],
            params[1] - lr * grads[1])

params = (jnp.zeros(X_j.shape[1]), 0.0)

# 🔥 Warm-up (important!)
for i in range(10):
    params = update(params, X_j, y_j, lr)

start = time.time()

for i in range(epochs):
    params = update(params, X_j, y_j, lr)

end = time.time()

jax_time = end - start

# Accuracy (train)
jax_preds_train = (predict(params, X_j) >= 0.5).astype(int)
jax_acc = jnp.mean(jax_preds_train == y_j)

# Test prediction
jax_preds_test = (predict(params, X_test_j) >= 0.5).astype(int)

submission_jax = pd.DataFrame({
    "PassengerId": test_df["PassengerId"],
    "Survived": np.array(jax_preds_test)
})

submission_jax.to_csv("/kaggle/working/submission_jax.csv", index=False)

# ============================================
# 3. FINAL COMPARISON OUTPUT
# ============================================

print("\n========== FINAL COMPARISON ==========")
print(f"Normal Accuracy : {normal_acc:.4f}")
print(f"JAX Accuracy    : {float(jax_acc):.4f}")

print(f"\nNormal Time     : {normal_time:.4f} sec")
print(f"JAX Time        : {jax_time:.4f} sec")

print("\nSpeedup (Normal / JAX):", normal_time / jax_time)
print("======================================")


========== FINAL COMPARISON ==========
Normal Accuracy : 0.6813
JAX Accuracy    : 0.6162

Normal Time     : 0.0399 sec
JAX Time        : 0.1710 sec

Speedup (Normal / JAX): 0.23334104903841862


In [15]:
import platform
import os

print("OS:", platform.system(), platform.release())
print("Processor:", platform.processor())
print("CPU cores:", os.cpu_count())


OS: Linux 6.6.113+
Processor: x86_64
CPU cores: 4
